# PascalVOC-SP: CPEN forward–backward stability

Node classification on native undirected PascalVOC-SP graphs ($n_0=14$, $m_0=2$, 21 node classes), with **simultaneous** node and boundary-edge objectives (`readout_mode='node+edge'`). The coupled-network / $\mu$P theory says hidden states $Z_t^{(\ell)}$ (particle stream $t=X$, edge stream $t=E$) stay $\Theta(1)$ in

$$
\rho_t^{(\ell)}=\frac{\|Z_t^{(\ell)}\|_F^2}{N_t D},
$$

and that a **width-transferred** Adam step $\eta=\eta_0/\sqrt{D}$ moves them by an amount that does not blow up as $D$ (and $L$) grow. $\Delta\rho$ is the same coordinate on $\Delta Z = Z_{\mathrm{after}}-Z_{\mathrm{init}}$.

$N_t$ is the number of **valid** tokens in that stream (unpadded superpixels, nonempty edges). The batch is treated as one token pool.

The loss matches training: weighted node CE on $W_X X^{(L)}$ (21 classes) plus weighted boundary-edge CE on $W_E E^{(L)}$ with targets $1[y_u\neq y_v]$ (2 classes). $T$ Adam steps are **on each minibatch**, not a full epoch.

**Layout.** A $3\times 3$ grid vs $\#$ trainable parameters:

- **Rows:** encoder $\ell=0$; first and last residual ($\ell=1$ open markers, $\ell=L$ filled); decoder $\ell=L{+}1$.
- **Columns:** init; after 1 Adam step; after $T=1000$ Adam steps.
- Particle traces are **blue circles**; edge traces are **orange squares**.
- Solid $=\rho$, dashed $=\Delta\rho$. Init has $\rho$ only.

Every point is a mean over several independent Pascal minibatches, with error bars and a $\pm 1$ std band.

**On $\ell=L{+}1$.** Unlike jets, both heads are token-wise: $z_X\in\mathbb{R}^{B\times N\times 21}$ and $z_E\in\mathbb{R}^{B\times M\times 2}$. We plot the same $\rho=\|Z\|_F^2/(N_t C_t)$ over valid tokens. The probe is only kept when `decoder_{x,e}(h^{(L)})$` matches the model outputs.

Helpers: `cpen.training.stability`.


In [1]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import torch

REPO = Path("../..").resolve() if (Path("../..") / "src" / "cpen").exists() else Path("..").resolve()
sys.path.insert(0, str(REPO / "src"))

import cpen.training.stability as stability

importlib.reload(stability)

from cpen.training.stability import (
    DEFAULT_ARCHS,
    DEFAULT_FIG_DIR,
    DEFAULT_N_ADAM_STEPS,
    DEFAULT_N_BATCHES,
    PASCAL_OUT_DIM,
    cuda_mem_line,
    load_pascal_batches,
    plot_layerwise_msq,
    plot_stability_vs_params,
    release_cuda,
    run_stability_ladder,
)
from cpen.training.transfer_plots import setup_mpl_style

setup_mpl_style()
FIG_DIR = DEFAULT_FIG_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
release_cuda()
print("stability:", stability.__file__)
print("device:", DEVICE)
print("archs:", DEFAULT_ARCHS)
print("FIG_DIR:", FIG_DIR)
if DEVICE.type == "cuda":
    print("cuda:", cuda_mem_line())
    alloc = torch.cuda.memory_allocated() / 2**30
    if alloc > 2.0:
        print(
            f"WARNING: this kernel already holds {alloc:.2f} GiB. "
            "Kernel → Restart, then run from the top. "
            "Also kill leftover ipykernel PIDs in nvidia-smi from earlier OOM runs."
        )


stability: /home/jdezoort/coupled-particle-edge-networks/src/cpen/training/stability.py
device: cuda
archs: ((2, 128), (3, 192), (4, 256), (5, 384), (6, 512))
FIG_DIR: /home/jdezoort/coupled-particle-edge-networks/notebooks/figures
cuda: NVIDIA A100 80GB PCIe MIG 1g.10gb  allocated=0.00 GiB  used=8.72/9.50 GiB


## Batches of PascalVOC-SP graphs

Several minibatches from the incidence-v2 cache (native undirected 2-edges). Each architecture is trained independently on each batch so the error band is batch-to-batch scatter. Model seed is shared; only the graph sample changes.


In [2]:
DATA_ROOT = Path("/scratch/gpfs/BHANIN/jdezoort/datasets")
N_GRAPHS = 4
N_BATCHES = DEFAULT_N_BATCHES

batches = load_pascal_batches(
    DATA_ROOT,
    n_graphs=N_GRAPHS,
    n_batches=N_BATCHES,
    split="train",
    seed=0,
    device=DEVICE,
)
for i, batch in enumerate(batches):
    n_valid = int(batch["mask"].sum())
    n_edge = int(batch["edge_mask"].sum())
    print(
        f"batch {i}: B={batch['x'].size(0)}  N={batch['x'].size(1)}  "
        f"M={batch['edge_x'].size(1)}  n0={batch['x'].size(-1)}  "
        f"m0={batch['edge_x'].size(-1)}  valid nodes={n_valid}  "
        f"valid edges={n_edge}"
    )


batch 0: B=4  N=500  M=1471  n0=14  m0=2  valid nodes=1921  valid edges=5475
batch 1: B=4  N=500  M=1471  n0=14  m0=2  valid nodes=1890  valid edges=5347
batch 2: B=4  N=500  M=1471  n0=14  m0=2  valid nodes=1965  valid edges=5552
batch 3: B=4  N=500  M=1471  n0=14  m0=2  valid nodes=1955  valid edges=5520


## Ladder

**CPEN**: transformer-like residuals, incidence operators, degree-norm, affine-free LN, dropout 0, `x_only_node_readout` so node CE is $W_X X^{(L)}$ only. The edge decoder is resized to 2 logits for the boundary head.

Adam uses the training map $\eta=\eta_0/\sqrt{D}$ and $\varepsilon=10^{-14}$. $T=1000$ steps are **on that batch**, not a full epoch — we are probing the parameterization, not measuring node macro-F1.

After a CUDA OOM, **restart the kernel** before re-running (IPython keeps the failed frame, so the GPU stays full).


In [ ]:
SMOKE = False  # True → 2 sizes, T=5, 2 batches; faster on login/CPU nodes

ETA_0 = 0.25
FAMILIES = ("cpen",)
SEED = 0

if SMOKE:
    ARCHS = ((2, 128), (4, 256))
    N_ADAM_STEPS = 5
    batches = batches[:2]
else:
    ARCHS = DEFAULT_ARCHS
    N_ADAM_STEPS = DEFAULT_N_ADAM_STEPS

print(
    f"families={FAMILIES}  archs={ARCHS}  η0={ETA_0}  T={N_ADAM_STEPS}  "
    f"n_batches={len(batches)}  smoke={SMOKE}  obj=node+edge  C={PASCAL_OUT_DIM}"
)
release_cuda()
if DEVICE.type == "cuda":
    print("cuda before ladder:", cuda_mem_line())

df = run_stability_ladder(
    batches,
    families=FAMILIES,
    archs=ARCHS,
    eta_0=ETA_0,
    n_adam_steps=N_ADAM_STEPS,
    seed=SEED,
    device=DEVICE,
    objective="node+edge",
    out_dim=PASCAL_OUT_DIM,
)
last = df.loc[
    df.apply(lambda r: r["stage"] == f"layer{int(r['depth']) - 1}", axis=1)
]
display(
    last.pivot_table(
        index=["family", "depth", "width", "n_params"],
        columns="token",
        values=["msq_init", "dmsq_1", "msq_T", "dmsq_T"],
        aggfunc="mean",
    )
)


families=('cpen',)  archs=((2, 128), (3, 192), (4, 256), (5, 384), (6, 512))  η0=0.25  T=1000  n_batches=4  smoke=False  obj=node+edge  C=21
cuda before ladder: NVIDIA A100 80GB PCIe MIG 1g.10gb  allocated=0.00 GiB  used=8.73/9.50 GiB
[stability] NVIDIA A100 80GB PCIe MIG 1g.10gb  allocated=0.00 GiB  used=8.73/9.50 GiB
[stability] batch 1/4  cpen          L=2 D=128  T=1000  obj=node+edge
[stability] batch 1/4  cpen          L=3 D=192  T=1000  obj=node+edge
[stability] batch 1/4  cpen          L=4 D=256  T=1000  obj=node+edge
[stability] batch 1/4  cpen          L=5 D=384  T=1000  obj=node+edge
[stability] batch 1/4  cpen          L=6 D=512  T=1000  obj=node+edge
[stability] batch 2/4  cpen          L=2 D=128  T=1000  obj=node+edge
[stability] batch 2/4  cpen          L=3 D=192  T=1000  obj=node+edge
[stability] batch 2/4  cpen          L=4 D=256  T=1000  obj=node+edge
[stability] batch 2/4  cpen          L=5 D=384  T=1000  obj=node+edge
[stability] batch 2/4  cpen          L=6 D=512  T

## $\rho$ and $\Delta\rho$ vs $\#$ parameters

If the theory is right:

- the encoder / residual init $\rho$ stays $O(1)$ across the ladder;
- the one-step $\Delta\rho$ does not grow with $D$;
- after $T$ steps, feature size is still $O(1)$ and the cumulative move stays controlled.

Particle = blue circles, edge = orange squares. Solid $\rho$, dashed $\Delta\rho$. On the residual row, open markers are $\ell=1$ and filled markers are $\ell=L$. Bands are mean $\pm$ 1 std over batches.


In [ ]:
plot_stability_vs_params(
    df,
    n_adam_steps=N_ADAM_STEPS,
    stem="pascal_stability",
    fig_dir=FIG_DIR,
    show=True,
)


## Layerwise check at one size

Same $\rho$, now vs layer index ($\ell=0$ encoder, $\ell=1,\ldots,L$ residuals, $\ell=L{+}1$ decoder). Residual $\mu$P should keep this from exploding in depth. Error bars are again batch-to-batch.


In [ ]:
# Prefer (4, 256) when present; otherwise the largest architecture in the run.
_archs = list(df[["depth", "width"]].drop_duplicates().itertuples(index=False, name=None))
DEPTH, WIDTH = (4, 256) if (4, 256) in _archs else _archs[-1]
plot_layerwise_msq(
    df,
    depth=DEPTH,
    width=WIDTH,
    ycol="msq_init",
    stem=f"pascal_stability_layers_L{DEPTH}_D{WIDTH}",
    fig_dir=FIG_DIR,
    show=True,
)
plot_layerwise_msq(
    df,
    depth=DEPTH,
    width=WIDTH,
    ycol="dmsq_1",
    stem=f"pascal_stability_layers_dZ_L{DEPTH}_D{WIDTH}",
    fig_dir=FIG_DIR,
    show=True,
)
